# Phugoid Oscillation

In the first lesson, _Phugoid Motion_, we described the physics of a glider's oscillatory trajectory, seen as an exchange of kinetic and potential energy. This analysis goes back to Frederick Lanchester, who published his book _"Aerodonetics"_ on aircraft stability in 1908. We concluded that first exposure to our problem of interest by plotting the flight paths predicted by Lanchester's analysis, known as _phugoids_.

Here, we will look at the situation when an aircraft is initially moving on the straight-line phugoid (obtained with the parameters $C=2/3$, $\cos\theta=1$, and $z=z_t$ in the previous analysis), and experiences a small upset, a wind gust that slightly perturbs its path. It will then enter into a gentle oscillation around the previous straight-line path: a _phugoid oscillation_.

As in the first lesson, $z$ is a depth measured positive downward from the chosen energy-reference level, and $z_t$ is its equilibrium value for trimmed flight. Thus $z$ is not the aircraft's altitude: an upward acceleration is $-d^2z/dt^2$. If we assume that the perturbation is small, then $\cos\theta=1$ is a good approximation and Newton's second law in the vertical direction is:

$$
\label{eq-phugoid-vertical-force}
L - W = - \frac{W}{g}\frac{d^2 z}{dt^2}
$$

We previously saw that the following relation holds for the ratio of lift to weight, in terms of the trim velocity $v_t$:

$$
\label{eq-phugoid-lift-weight-ratio}
\frac{L}{W}=\frac{v^2}{v_t^2}
$$

This will be useful: we can divide [Equation %s](#eq-phugoid-vertical-force) by the weight and use [Equation %s](#eq-phugoid-lift-weight-ratio) to replace $L/W$. Another useful relation from the previous lesson expressed the conservation of energy (per unit mass) as $v^2 = 2 gz$. With this, [Equation %s](#eq-phugoid-vertical-force) is rearranged as:

$$
\label{eq-linear-phugoid-ode}
\frac{d^2z}{dt^2} + \frac{gz}{z_t} = g
$$

Look at [Equation %s](#eq-linear-phugoid-ode) for a moment. Does it ring a bell? Do you recognize it?

If you remember from your physics courses the equation for _simple harmonic motion_, you should see the similarity! 

Take the case of a simple spring. Hooke's law is $F=-kx$, where $F$ is a restoring force, $x$ the displacement from a position of equilibrium and $k$ the spring constant. This results in the following ordinary differential equation for the displacement:

$$
\begin{equation}
 \frac{d^2 x}{dt^2}= -\frac{k}{m}x
\end{equation}
$$

which has the solution $x(t) = A \cos(\omega t- \phi)$, representing simple harmonic motion with an angular frequency $\omega=\sqrt{k/m}=2\pi f$ and phase angle $\phi$.

Now look back at [Equation %s](#eq-linear-phugoid-ode): it has nearly the same form and it represents simple harmonic motion with angular frequency $\omega=\sqrt{g/z_t}$ around the equilibrium depth $z_t$. 

Think about this for a moment ... we can immediately say what the period of the oscillation is: exactly $2 \pi \sqrt{z_t/g}$ — or, in terms of the trim velocity, $\pi \sqrt{2} v_t/g$.

_This is a remarkable result!_ Think about it: we know nothing about the aircraft, or the flight altitude, yet we can obtain the period of the phugoid oscillation simply as a function of the trim velocity. For example, if trim velocity is 200 knots, we get a phugoid period of about 47 seconds—over that time, you really would not notice anything if you were flying in that aircraft.

Next, we want to be able to compute the trajectory of the aircraft for a given initial perturbation. We will do this by numerically integrating the equation of motion.

## Prepare to integrate

We want to integrate the differential equation and plot the trajectory of the aircraft. Are you ready?

:::{warning .simple .dropdown icon=false open=false} On paper
This is an important modeling approach that makes the mathematics better adapted to computation: formulate a second-order differential equation as a system of first-order equations, written in vector form. The vector computations are then naturally handled by arrays. Be sure to follow this derivation on paper, taking your own notes. 
:::

[Equation %s](#eq-linear-phugoid-ode) is a second-order ordinary differential equation (ODE). Let's represent the time derivative with a prime and write it like this:

$$
\begin{equation}
z''(t) + \frac{g \,z(t)}{z_t}=g
\end{equation}
$$

There's a convenient trick when we work with ODEs: we can turn this 2nd-order equation into a system of two 1st-order equations introducing an intermediate variable for the first derivative. Like this:

$$
\label{eq-linear-phugoid-system}
\begin{aligned}
z'(t) &= b(t)\\
b'(t) &= g\left(1-\frac{z(t)}{z_t}\right)
\end{aligned}
$$

Here $b=z'$ is the rate of change of depth: $b>0$ means downward velocity, while $b<0$ means upward velocity. 

Another way to look at a system of two 1st-order ODEs is by using vectors. You can make a vector with the two state variables, 

$$
\begin{equation}
\vec{u}  = \begin{bmatrix} z \\ b \end{bmatrix}
\end{equation}
$$

and write the differential system as a single vector equation:

$$
\begin{equation}
\vec{u}'(t)  = \begin{bmatrix} b\\ g-g\frac{z(t)}{z_t} \end{bmatrix}
\end{equation}
$$

If you call the right-hand-side $\vec{f}(\vec{u})$, then the equation is very short: $\vec{u}'(t) = \vec{f}(\vec{u})$—but let's drop those arrows to denote vectors from now on, as they are a bit cumbersome: just remember that $u$ and $f$ are vectors in the phugoid equation of motion.

Next, we'll prepare to solve this problem numerically.

## Initial value problems

Let's step back for a moment. Suppose we have a first-order ODE $u'=f(u)$. You know that if we were to integrate this, there would be an arbitrary constant of integration. To find its value, we do need to know one point on the curve $(t, u)$. When the derivative in the ODE is with respect to time, we call that point the _initial value_ and write something like this:

$$
u(t=0)=u_0
$$

In the case of a second-order ODE, we already saw how to write it as a system of first-order ODEs, and we would need an initial value for each equation: two conditions are needed to determine our constants of integration. The same applies for higher-order ODEs: if it is of order $n$, we can write it as $n$ first-order equations, and we need $n$ known values. If we have that data, we call the problem an _initial value problem_.

Remember the definition of a derivative? The derivative represents the slope of the tangent at a point of the curve $u=u(t)$, and the definition of the derivative $u'$ for a function is:

$$
u'(t) = \lim_{\Delta t\rightarrow 0} \frac{u(t+\Delta t)-u(t)}{\Delta t}
$$

If the step $\Delta t$ is already very small, we can _approximate_ the derivative by dropping the limit. We can write:

$$
\label{eq-forward-time-step}
u(t+\Delta t) \approx u(t) + u'(t) \Delta t
$$

With [Equation %s](#eq-forward-time-step), and because we know $u'(t)=f(u)$, if we have an initial value, we can step by $\Delta t$ and find the value of $u(t+\Delta t)$, then we can take this value, and find $u(t+2\Delta t)$, and so on: we say that we _step in time_, numerically finding the solution $u(t)$ for a range of values: $t_1, t_2, t_3 \cdots$, each separated by $\Delta t$. The numerical solution of the ODE is simply the table of values $t_i, u_i$ that results from this process.

## Discretization

:::{warning .simple .dropdown icon=false open=false} In your notebook

Create a new, clean notebook for your work, rather than executing the code cells in this lesson notebook. Keep the lesson open as a reference and reconstruct **all** of the code presented here: the time grid, the Forward Euler updates, the trajectory plots, the exact solution, and the refinement study. Type the numerical method yourself so that every line can be traced back to an equation in the lesson; copying mechanical details such as imports and plot labels is fine.

As you reconstruct, make the small changes requested in the local exercises and compare each result with your expectations before continuing. After completing the refinement study, carry out the final refactoring challenge: create a function that implements Euler's method, then rewrite the calculation for multiple time-step sizes to call your function instead of repeating the time-integration loop. Consult [Reconstruct a lesson](../../appendices/notebook-workflow.md#notebook-reconstruct) for the general workflow.
:::

In order to execute the process described above and find the numerical solution of the ODE, we start by choosing the values $t_1,t_2,t_3 \cdots t_n$—we call these values our *grid* in time. The first point of the grid is given by our _initial value_, and the small difference between two consecutive times is called the _time step_, denoted by $\Delta t$.  The solution value at time $t_n$ is denoted by $u_n$.

Let's build a time grid for our problem. We first choose a final time $T$ and the time step $\Delta t$. In code, we'll use readily identifiable variable names: `T` and `dt`, respectively. With those values set, we calculate `num_steps`, the number of updates needed to reach $T$. Because the grid includes both the initial and final times, it contains `num_steps + 1` points.

Let's write some code. The first thing we do in Python is load two scientific-Python libraries: NumPy for numerical functions and arrays, and Matplotlib for plotting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

Now initialize `T` and `dt`, calculate `num_steps`, and build a NumPy array containing the `num_steps + 1` time points in the grid.

In [ ]:
# Create the time grid.
T = 100.0  # length of the time interval
dt = 0.02  # time-step size
num_steps = int(T / dt)  # number of time steps
t = np.linspace(0.0, T, num=num_steps + 1)  # time grid

We have our time grid! Now it's time to apply the numerical time stepping represented by [Equation %s](#eq-forward-time-step).

:::{note} Python refresher — `linspace` and `arange`
:icon: false

`int()` converts a number to an integer; here $T/\Delta t$ is a whole number by construction. `np.linspace(start, stop, num=...)` returns a requested number of evenly spaced points and includes both endpoints. By contrast, `np.arange(start, stop, step)` advances by a requested step and normally excludes `stop`. In your clean notebook, rebuild this grid with `np.arange()` and check both the number of points and the final time. Because the step size is a `float` you should check the endpoint rather than assuming.

**Pro tip:** Enter a question mark followed by any function, e.g., `?np.linspace`, into a code cell and execute it to open its help in Jupyter.
:::

## Euler's method

The approximate solution at time $t_n$ is $u_n$, and the numerical solution of the differential equation consists of computing a sequence of approximate solutions by the following formula, based on [Equation %s](#eq-forward-time-step):

$$
\label{eq-forward-euler}
u_{n+1} = u_n + \Delta t \,f(u_n)
$$

[Equation %s](#eq-forward-euler) is called **Euler's method**.

Applying [Equation %s](#eq-forward-euler) to the phugoid system in [Equation %s](#eq-linear-phugoid-system) gives the following algorithm that we need to implement in code:

$$
\label{eq-phugoid-forward-euler}
\begin{aligned}
z_{n+1} & = z_n + \Delta t \, b_n \\
b_{n+1} & = b_n + \Delta t \left(g - \frac{g}{z_t} \, z_n \right)
\end{aligned}
$$

### And solve!

To implement [Equation %s](#eq-phugoid-forward-euler), we need to set things up in code: define the parameter values needed in the model, initialize a NumPy array to hold the two state variables, and initialize another array for the depth-coordinate values.

In [ ]:
# Set the model parameters and initial conditions.
z_0 = 100.0  # initial depth below the energy reference
b_0 = 10.0   # initial downward velocity from a gust or downdraft
z_t = 100.0  # equilibrium depth for trimmed flight
g = 9.81     # acceleration due to gravity

# Set the initial value of the numerical solution.
u = np.array([z_0, b_0])

# Create an array to store the depth coordinate at each grid point.
z = np.zeros(num_steps + 1)
z[0] = z_0

:::{note} Python refresher — arrays, indexing, and unpacking
:icon: false

The decimal points make the parameter values floating-point numbers. `np.array([z_0, b_0])` collects the two initial state variables in the vector `u`, while `np.zeros(num_steps + 1)` creates the storage array and initializes every entry to zero. Python starts indexing at zero, so `z[0] = z_0` stores the initial depth. In the loop below, `z_n, b_n = u` unpacks the two entries into names that match [Equation %s](#eq-phugoid-forward-euler).
:::

Now we can step in time using Euler's method. `range(num_steps)` supplies the indices from `0` through `num_steps - 1`; each pass advances both state variables and stores the new depth at index `n + 1`.

In [ ]:
# Temporal integration using Euler's method.
for n in range(num_steps):
    z_n, b_n = u
    du_dt = np.array([b_n, g * (1.0 - z_n / z_t)])
    u = u + dt * du_dt
    z[n + 1] = u[0]

Make sure you understand what this code is doing. This is a basic pattern in numerical methods: iterations in a time variable that apply a numerical scheme at each step.

## Plot the vertical displacement

If the code is correct, we have stored the downward-positive depth coordinate in the array `z`. For a picture of the aircraft's motion in the sky, however, we expect the vertical axis to point upward. We therefore define the vertical displacement from trim as

$$
\label{eq-vertical-displacement}
\eta(t)=z_t-z(t).
$$

[Equation %s](#eq-vertical-displacement) defines $\eta$ as positive above the trim level and negative below it. This change does not alter the numerical solution: Euler's method still advances $z$ and $b$, and we convert from $z$ to $\eta$ only for visualization. In particular, a positive $b_0$ makes $z$ increase and $\eta$ decrease, so a downdraft appears as downward motion on the plot.

You should explore the [Matplotlib Pyplot tutorial](https://matplotlib.org/stable/tutorials/pyplot.html) (if you need to) and familiarize yourself with the tools that control the size, labels, line style, and so on. Creating good plots is a useful skill: it is about communicating your results effectively.

:::{note} Python refresher — publication-quality figures
:icon: false

Matplotlib allows fine-tuning of many plot features to achieve publication-quality figures. the [`plt.rcParams`](https://matplotlib.org/stable/api/matplotlib_configuration_api.html#matplotlib.rcParams) dictionary holds default styling settings; changing it once applies the chosen font to subsequent figures. Here, we are setting the font family to `serif`, and the size to 16 pt. [`fig.tight_layout()`](https://matplotlib.org/stable/users/explain/axes/tight_layout_guide.html) adjusts the spacing so that labels are not clipped. Matplotlib has many ways to specify [colors](https://matplotlib.org/stable/gallery/color/color_demo.html)
:::

Here, we set the figure size, vertical limits, grid, and axis labels. The final `ax.plot()` call draws a continuous red line.

In [ ]:
# Set the font family and size to use for Matplotlib figures.
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

# Convert depth to vertical displacement, positive upward.
eta = z_t - z

# Plot the vertical displacement from trim.
fig, ax = plt.subplots(figsize=(8, 3))
ax.set_title('Phugoid displacement from trim')
ax.set_xlabel('Time [s]')
ax.set_ylabel(r'Vertical displacement, $\eta$ [m]')
ax.set_xlim(t[0], t[-1])
ax.set_ylim(-70.0, 70.0)
ax.grid()
ax.plot(t, eta, color='tab:red', linestyle='-', linewidth=2)
fig.tight_layout()

:::{warning .simple .dropdown icon=false open=false} In your notebook

Try changing the value of `b_0` in the initial conditions. Remember that its sign follows the depth coordinate: positive is downward and negative is upward.  Study these cases:

* What happens with a stronger downdraft ($b_0>0$)?  
* What happens with an updraft ($b_0<0$)?  
* What happens if there is no initial vertical perturbation ($b_0=0$)?
:::

## Exact solution

The equation for phugoid oscillations is a 2nd-order, linear ODE and it has an exact solution of the following form:

$$
\label{eq-phugoid-general-solution}
z(t) = A \sin \left(\sqrt{\frac{g}{z_t}} t \right) + B \cos \left(\sqrt{\frac{g}{z_t}} t \right) + z_t
$$

where $A$ and $B$ are constants that we solve for using initial conditions.  

Our numerical solution used the initial conditions:

$$
\label{eq-phugoid-initial-conditions}
\begin{aligned}
z(0) &= z_0 \\
b(0) &= b_0
\end{aligned}
$$

Applying the initial conditions in [Equation %s](#eq-phugoid-initial-conditions) to [Equation %s](#eq-phugoid-general-solution) and solving for $A$ and $B$ gives:

$$
\label{eq-phugoid-exact-solution}
z(t) = b_0 \sqrt{\frac{z_t}{g}} \sin \left(\sqrt{\frac{g}{z_t}} t \right) + (z_0-z_t) \cos \left(\sqrt{\frac{g}{z_t}} t \right) + z_t
$$

We already defined all of the variables in [Equation %s](#eq-phugoid-exact-solution), so we can immediately compute the exact depth coordinate $z_{\rm exact}$. We then apply the sky-view transformation in [Equation %s](#eq-vertical-displacement), $\eta_{\rm exact}=z_t-z_{\rm exact}$, for plotting.

:::{note} Python refresher — NumPy functions and long expressions
:icon: false

NumPy's `np.sqrt()`, `np.sin()`, and `np.cos()` functions act element by element when their input is an array. Thus applying the trigonometric functions to `t` produces the exact solution at every time in the grid. For a long expression, enclose it in parentheses and insert line breaks; Python will treat the indented lines as one expression.
:::

In [ ]:
omega = np.sqrt(g / z_t)
z_exact = (
    b_0 / omega * np.sin(omega * t)
    + (z_0 - z_t) * np.cos(omega * t)
    + z_t
)
eta_exact = z_t - z_exact

## Compare with the exact solution

Now we can plot both the numerical and exact vertical displacements to see how well Euler's method approximated the phugoid oscillation. Both curves use $\eta$, so upward and downward on the graph match the aircraft's motion in the sky.

To add another curve to a plot, call `ax.plot()` a second time on the same Axes. Giving each curve a `label` lets `ax.legend()` identify them.

In [ ]:
# Plot the numerical and exact vertical displacements.
fig, ax = plt.subplots(figsize=(8, 3))
ax.set_title('Phugoid displacement from trim')
ax.set_xlabel('Time [s]')
ax.set_ylabel(r'Vertical displacement, $\eta$ [m]')
ax.set_xlim(t[0], t[-1])
ax.set_ylim(-70.0, 70.0)
ax.grid()
ax.plot(
    t, eta, label='Numerical',
    color='tab:red', linestyle='-', linewidth=2,
)
ax.plot(
    t, eta_exact, label='Exact',
    color='tab:grey', linestyle='-', linewidth=2,
)
ax.legend()
fig.tight_layout()

The two curves agree fairly well at first, but the numerical oscillation grows toward the end. That growth is a numerical artifact: the ideal phugoid is undamped, so its exact amplitude remains constant. Re-run the previous steps with a smaller time step, say $dt=0.01$, and notice that the artificial growth becomes slower.

This brings up two different features of numerical methods. A method is **convergent** if, over a fixed time interval, its numerical solution approaches the exact solution as the time-step size tends to zero. If the global error behaves like $E=O(\Delta t^p)$, the method has order $p$; _Forward Euler is first-order convergent._

**Consistency** is a related local property. Substitute the exact solution into one step of the numerical method and examine the residual. For Forward Euler, the unnormalized one-step defect is $O(\Delta t^2)$, so the residual per unit time is $O(\Delta t)$ and tends to zero. _Forward Euler is therefore consistent._ Consistency alone does not guarantee convergence: the method must also control how errors grow from one step to the next.

Convergence on a fixed interval does not guarantee faithful behavior over arbitrarily long times. For this undamped oscillator, Forward Euler adds a small amount of numerical energy at every step, causing the amplitude to grow. Reducing $\Delta t$ slows that growth, but no nonzero time step eliminates it completely. The refinement study below asks the fixed-interval question: does the global error tend to zero, and at what rate, as $\Delta t$ decreases?

:::{warning .simple .dropdown icon=false open=false} On paper

Derive the order of convergence of Forward Euler before examining the numerical errors.

**Taylor-series reminder.** If $u(t)$ is sufficiently smooth, its value one step away can be expanded about $t_n$ as

$$
\label{eq-taylor-one-step}
u(t_n+\Delta t)
=u(t_n)+\Delta t\,u'(t_n)
+\frac{\Delta t^2}{2}u''(t_n)+O(\Delta t^3).
$$

The notation $O(\Delta t^q)$ collects terms whose magnitude is bounded by a constant times $\Delta t^q$ as $\Delta t\to0$. Use [Equation %s](#eq-taylor-one-step) to complete the following argument:

1. Substitute the differential equation $u'=f(u)$ into the Taylor expansion.
2. Compare the exact Taylor step with the Forward Euler step in [Equation %s](#eq-forward-euler). Define the **one-step defect** as

   $$
   \label{eq-forward-euler-one-step-defect}
   d_{n+1}=u(t_{n+1})-\left[u(t_n)+\Delta t\,f(u(t_n))\right].
   $$

3. Identify the leading omitted Taylor term and show the order of $d_{n+1}$.
4. A fixed interval of length $T$ contains $N=T/\Delta t$ steps. Assuming errors remain controlled as they propagate across this smooth linear problem, estimate the global error—up to a factor independent of $\Delta t$—by multiplying the size of one defect by $N$. Simplify the resulting power of $\Delta t$.
5. State separately the order of the one-step defect and the order of the accumulated global error. Does your result agree with the statement that Forward Euler is first-order convergent?

Keep this derivation beside the log-log error plot below: it supplies the expected slope independently of the computation.
:::

## Convergence

To compare the two solutions, we need to use a **norm** of the difference, like the $L_1$ norm, for example.

$$
\label{eq-discrete-l1-error}
E = \Delta t \sum_{n=0}^N \left|z(t_n) - z_n\right|
$$

[Equation %s](#eq-discrete-l1-error) sums the individual differences between the exact and numerical solutions at the mesh points. In other words, $E$ is a discrete representation of the integral over the interval $T$ of the absolute difference between the computed $z$ and $z_{\rm exact}$:

$$
\label{eq-continuous-l1-error}
E = \int \vert z-z_{\rm exact}\vert dt
$$

Although [Equation %s](#eq-continuous-l1-error) is written using the depth coordinate $z$, we plotted the upward-positive displacement $\eta$. Subtracting the same $z_t$ and reversing the sign does not change an absolute difference: $|\eta-\eta_{\rm exact}|=|z-z_{\rm exact}|$.

We check for convergence by calculating the numerical solution using progressively smaller values of `dt`. We already have most of the code that we need.  We just need to add an extra loop and an array of different $\Delta t$ values to iterate through.

:::{warning} Runtime

The cell below can take a little while to finish (the last $\Delta t$ value alone requires 1 million iterations!). If the cell is still running, the input label will say `In [*]`. When it finishes, the `*` will be replaced by a number.
:::

In [ ]:
# Set the list of time-step sizes.
dt_values = [0.1, 0.05, 0.01, 0.005, 0.001, 0.0001]

# Create an empty list for the depth-coordinate solution on each grid.
z_solutions = []

for dt_trial in dt_values:
    num_steps_trial = int(T / dt_trial)
    t_trial = np.linspace(0.0, T, num=num_steps_trial + 1)
    # Set the initial conditions.
    u_trial = np.array([z_0, b_0])
    z_trial = np.zeros(num_steps_trial + 1)
    z_trial[0] = z_0
    # Temporal integration using Euler's method.
    for n in range(num_steps_trial):
        z_n, b_n = u_trial
        du_dt = np.array([b_n, g * (1.0 - z_n / z_t)])
        u_trial = u_trial + dt_trial * du_dt
        z_trial[n + 1] = u_trial[0]
    z_solutions.append(z_trial)

### Calculate the error

We now have a numerical depth-coordinate solution for each $\Delta t$ in the list `z_solutions`. The list method `.append()` added each completed NumPy array without requiring us to know its length in advance. To calculate the error corresponding to each $\Delta t$, we can write a function.

In [ ]:
def l1_error(z_numerical, z_exact, dt):
    '''Return the discrete L1 error in the numerical depth coordinate.
    
    Parameters
    ----------
    z_numerical : np.ndarray
        The numerical depth coordinate as an array of floats.
    z_exact : np.ndarray
        The exact depth coordinate as an array of floats.
    dt : float
        The time-step size.
        
    Returns
    -------
    error : float
        Discrete L1 error with respect to the exact solution.
    '''
    return dt * np.sum(np.abs(z_numerical - z_exact))

:::{note} Python refresher — element-wise operations and reductions
:icon: false

In `z_numerical - z_exact`, NumPy subtracts corresponding elements rather than subtracting only one pair of values. `np.abs()` likewise takes the absolute value of every element, and `np.sum()` then reduces the resulting array to one number. Together these operations translate [Equation %s](#eq-discrete-l1-error) directly into code. Here is a small example of element-wise subtraction:

:::

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 4, 4])

b - a

Now, we iterate through each $\Delta t$ value and calculate the corresponding error.
In the following code cell, we use the built-in function [`zip`](https://docs.python.org/3/library/functions.html#zip) to pair each array in `z_solutions` with its time-step size in `dt_values`. The trial-specific names keep this loop from overwriting the baseline variables `t`, `dt`, and `z_exact` used earlier in the lesson.

In [ ]:
# Create an empty list to store the errors on each time grid.
error_values = []

for z_trial, dt_trial in zip(z_solutions, dt_values):
    num_steps_trial = int(T / dt_trial)
    t_trial = np.linspace(0.0, T, num=num_steps_trial + 1)
    # Compute the exact depth-coordinate solution.
    z_exact_trial = (
        b_0 / omega * np.sin(omega * t_trial)
        + (z_0 - z_t) * np.cos(omega * t_trial)
        + z_t
    )
    # Calculate the L1-norm of the error for the present time grid.
    error_values.append(l1_error(z_trial, z_exact_trial, dt_trial))

Remember, *if* the method is convergent then the error should get smaller as $\Delta t$ gets smaller. To visualize this trend across several scales, we use `ax.loglog()` to make both axes logarithmic. A relationship $E\propto\Delta t^p$ then appears as a straight line with slope $p$.

Our paper derivation predicts $p=1$, so we also plot a reference line proportional to $\Delta t$. Its vertical placement is fixed by the error on the finest grid; only its slope carries meaning. In Python, an index of `-1` selects the last item, and converting `dt_values` to a NumPy array lets us scale all its entries at once. The equal aspect setting makes one decade occupy the same visual distance on each axis, so a slope-one line appears at 45 degrees. We are comparing the computed curve visually with the expected order, not calculating an observed value of $p$.

In [ ]:
# Construct a first-order reference line through the finest-grid error.
dt_array = np.array(dt_values)
first_order_reference = error_values[-1] * dt_array / dt_array[-1]

# Plot the error versus the time-step size.
fig, ax = plt.subplots(figsize=(5.0, 5.0))
ax.set_title(r'$L_1$ error vs. time-step size')
ax.set_xlabel(r'$\Delta t$')
ax.set_ylabel('Error')
ax.grid()
ax.loglog(
    dt_array, error_values, label='Forward Euler error',
    color='tab:red', linestyle='--', marker='o',
)
ax.loglog(
    dt_array, first_order_reference, label=r'Expected $O(\Delta t)$',
    color='tab:blue', linestyle='-',
)
ax.set_aspect('equal', adjustable='box')
ax.legend()
fig.tight_layout()

This is the kind of result we like to see: as $\Delta t$ shrinks toward the left, the error decreases. Over the finer step sizes, the computed curve is approximately parallel to the $O(\Delta t)$ reference line, visually supporting the first-order result derived on paper.

**Reconstruction checkpoint.** Your clean notebook should now contain every calculation in this lesson. Complete the refactoring challenge from the opening **In your notebook** admonition: put the Euler integration in a function and use that function for the refinement study. Check that the refactored version reproduces the trajectory and error plot before treating it as a successful reconstruction.